# LeetCode 909 — Snakes and Ladders bằng Python

Link đề: https://leetcode.com/problems/snakes-and-ladders/ · Độ khó: **Medium** · Chủ đề: Array (mảng), Breadth-First Search (BFS), Matrix (ma trận)

Skeleton LeetCode:

```python
class Solution:
    def snakesAndLadders(self, board: List[List[int]]) -> int:
        ...
```

Ghi chú: bài toán này là **tìm đường đi ngắn nhất trong đồ thị không trọng số**: mỗi ô là một đỉnh, mỗi nước đi (một lần tung xúc xắc) là một cạnh, và ta cần số cạnh ít nhất từ ô `1` tới ô `n^2`. Khó khăn chính nằm ở việc **chuyển nhãn ô sang tọa độ** của bảng theo kiểu **zig-zag (boustrophedon)** và xử lý đúng quy tắc **chỉ đi theo rắn/thang một lần mỗi nước đi**.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: bảng `n x n` đánh số `1..n^2` kiểu zig-zag từ góc dưới trái, mỗi lượt đi tới một ô trong `[curr + 1, min(curr + 6, n^2)]`; nếu ô đó có rắn/thang thì **bắt buộc** đi tới đích của nó (chỉ một lần); tìm số lượt tối thiểu để tới `n^2` hoặc `-1`.
2. Nắm các công cụ Python cần thiết: `collections.deque`, phép `divmod`, mảng khoảng cách/`visited`, "làm phẳng" bảng thành mảng 1 chiều, và hàm chuyển nhãn ô thành `(hàng, cột)`.
3. Hiểu nguyên lý: **BFS** cho đường đi ngắn nhất khi mọi cạnh có độ dài `1`; mỗi ô chỉ cần thăm một lần (lần đầu tiên tới là nhanh nhất).
4. Đi qua 4 cách: lan truyền khoảng cách nhiều vòng kiểu Bellman-Ford (brute, đúng định nghĩa), BFS với hàm đổi nhãn thành tọa độ, BFS trên mảng đã làm phẳng (cách nộp chính) và BFS theo từng lớp bằng tập hợp; đối chiếu với hai oracle độc lập.
5. Tránh các lỗi kinh điển: đi theo chuỗi thang liên tiếp trong một lượt, đổi nhãn sang tọa độ sai ở các hàng đảo chiều, dùng tham lam hoặc DFS thay cho BFS, và quên trả `-1` khi không tới được.

## 2. Đề bài & ví dụ

Cho ma trận `board` kích thước `n x n`. Các ô được đánh số `1..n^2` theo kiểu **zig-zag (boustrophedon)** bắt đầu từ góc **dưới trái** `board[n - 1][0]` và **đổi chiều mỗi hàng**.

Bạn bắt đầu ở ô `1`. Mỗi lượt, từ ô `curr`:

- chọn ô đích `next` có nhãn trong `[curr + 1, min(curr + 6, n^2)]` (mô phỏng xúc xắc 6 mặt: luôn có tối đa 6 lựa chọn);
- nếu `next` có rắn hoặc thang thì **phải** di chuyển tới đích của nó; ngược lại di chuyển tới `next`;
- trò chơi kết thúc khi tới ô `n^2`.

Ô ở hàng `r`, cột `c` có rắn/thang nếu `board[r][c] != -1`, đích là `board[r][c]`. Ô `1` và `n^2` không phải điểm bắt đầu của rắn/thang nào.

Lưu ý: mỗi lượt chỉ đi theo rắn/thang **tối đa một lần**; nếu đích của rắn/thang lại là điểm bắt đầu của rắn/thang khác thì **không** đi tiếp. Ví dụ `board = [[-1,4],[-1,3]]`: lượt đầu chọn ô `2`, đi theo thang tới ô `3` nhưng **không** đi tiếp thang tới `4`.

Trả về số lượt tung xúc xắc **ít nhất** để tới ô `n^2`, hoặc `-1` nếu không thể.

**Ví dụ 1**

- Input: `board = [[-1,-1,-1,-1,-1,-1],[-1,-1,-1,-1,-1,-1],[-1,-1,-1,-1,-1,-1],[-1,35,-1,-1,13,-1],[-1,-1,-1,-1,-1,-1],[-1,15,-1,-1,-1,-1]]`
- Output: `4`
- Giải thích: `1 -> 2` (thang lên `15`), `15 -> 17` (rắn xuống `13`), `13 -> 14` (thang lên `35`), `35 -> 36`.

**Ví dụ 2**

- Input: `board = [[-1,-1],[-1,3]]`
- Output: `1`

**Ràng buộc:**

- `n == board.length == board[i].length`
- `2 <= n <= 20`
- `board[i][j]` là `-1` hoặc nằm trong `[1, n^2]`.
- Các ô `1` và `n^2` không phải điểm bắt đầu của rắn/thang.

In [1]:
from typing import List
from collections import deque
import random, time

examples = [
    ([[-1, -1, -1, -1, -1, -1], [-1, -1, -1, -1, -1, -1], [-1, -1, -1, -1, -1, -1],
      [-1, 35, -1, -1, 13, -1], [-1, -1, -1, -1, -1, -1], [-1, 15, -1, -1, -1, -1]], 4),
    ([[-1, -1], [-1, 3]], 1),
]
for board, expected in examples:
    print(f"n={len(board)} kỳ vọng={expected}")

n=6 kỳ vọng=4
n=2 kỳ vọng=1


## 3. Kiến thức nền

### 3.1. Đánh số zig-zag và đổi nhãn thành tọa độ

Hàng đáy (chỉ số `n - 1`) được đánh số từ **trái sang phải**: `1..n`. Hàng kế phía trên đánh số **từ phải sang trái**, rồi lại từ trái sang phải, và cứ thế luân phiên.

Với nhãn `s` (bắt đầu từ `1`), đặt `q, rem = divmod(s - 1, n)`:

- `q` là số hàng tính từ đáy (`0` là hàng đáy), nên hàng của bảng là `r = n - 1 - q`;
- nếu `q` **chẵn** (đi trái sang phải) thì cột `c = rem`;
- nếu `q` **lẻ** (đi phải sang trái) thì cột `c = n - 1 - rem`.

In [2]:
def label_to_rc(s: int, n: int):
    q, rem = divmod(s - 1, n)
    r = n - 1 - q
    c = rem if q % 2 == 0 else n - 1 - rem
    return r, c

n = 4
grid = [[0] * n for _ in range(n)]
for s in range(1, n * n + 1):
    r, c = label_to_rc(s, n)
    grid[r][c] = s
for row in grid:
    print(" ".join(f"{x:2}" for x in row))

16 15 14 13
 9 10 11 12
 8  7  6  5
 1  2  3  4


### 3.2. BFS trên đồ thị không trọng số

BFS dùng `deque`: lấy phần tử đầu (`popleft`), duyệt mọi láng giềng chưa thăm, đánh dấu và đẩy vào cuối. Vì mọi cạnh có độ dài `1`, **lần đầu** một đỉnh được tới là qua **số cạnh ít nhất**, nên khoảng cách `dist[v] = dist[u] + 1` là tối ưu.

In [3]:
q = deque([1, 2, 3])
q.append(4)
print(q.popleft(), list(q))

1 [2, 3, 4]


### 3.3. Làm phẳng bảng thành mảng 1 chiều

Thay vì đổi nhãn thành tọa độ mỗi lần, ta tạo mảng `flat` độ dài `n^2 + 1` trong đó `flat[s]` là giá trị của ô nhãn `s` (đã đảo chiều các hàng lẻ). Khi đó "ô `s` có rắn/thang không" chỉ cần `flat[s] != -1`.

In [4]:
def flatten(board: List[List[int]]) -> List[int]:
    n = len(board)
    flat = [0]  # bỏ chỉ số 0 để nhãn bắt đầu từ 1
    for q, row in enumerate(reversed(board)):          # q = 0 là hàng đáy
        flat.extend(row if q % 2 == 0 else row[::-1])
    return flat

print(flatten(examples[1][0]), "| board =", examples[1][0])
big = flatten(examples[0][0])
print("ô 2 ->", big[2], ", ô 17 ->", big[17], ", ô 14 ->", big[14])

[0, -1, 3, -1, -1] | board = [[-1, -1], [-1, 3]]
ô 2 -> 15 , ô 17 -> 13 , ô 14 -> 35


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Ví dụ 2: `board = [[-1,-1],[-1,3]]`, `n = 2`. Các ô: `1` ở `(1,0)`, `2` ở `(1,1)`, `3` ở `(0,1)`, `4` ở `(0,0)`. Ô `2` có thang tới `3`. Từ ô `1` ta chọn `next` trong `[2, min(1 + 6, 4)] = [2, 4]`, tức các ô `2, 3, 4`: chọn ngay ô `4 = n^2` ở lượt đầu. Đáp án `1`.

Ví dụ 1: ta tìm đường đi qua thang/rắn để rút ngắn: `1 -> 2 (thang 15) -> 17 (rắn 13) -> 14 (thang 35) -> 36`. 4 lượt, đúng đáp án.

Nhìn như đồ thị: cạnh từ `s` tới `f(s + d)` với `d = 1..6` (`f(x)` là đích sau rắn/thang nếu ô `x` có, ngược lại là `x`). Số lượt tối thiểu là **khoảng cách ngắn nhất** từ `1` tới `n^2`.

## 5. Cách 1 — Brute Force: lan truyền khoảng cách nhiều vòng (Bellman-Ford)

**Ý tưởng.** Không dùng thứ tự BFS: gán `dist[1] = 0`, `dist` các ô khác `vô cùng`; lặp tối đa `n^2` vòng, mỗi vòng thử **mọi** ô `s` đã có khoảng cách và mọi `d = 1..6`: cập nhật `dist[f(s + d)] = min(dist[f(s + d)], dist[s] + 1)`; dừng khi một vòng không còn thay đổi. Đây là cách giải bài toán đường ngắn nhất theo định nghĩa nên cũng làm **oracle**.

In [5]:
INF = float("inf")

def snakes_bellman_ford(board: List[List[int]]) -> int:
    n = len(board)
    N = n * n
    flat = flatten(board)
    dist = [INF] * (N + 1)
    dist[1] = 0
    for _ in range(N):                                   # tối đa N vòng
        changed = False
        for s in range(1, N):
            if dist[s] == INF:
                continue
            for nxt in range(s + 1, min(s + 6, N) + 1):
                dest = flat[nxt] if flat[nxt] != -1 else nxt
                if dist[s] + 1 < dist[dest]:
                    dist[dest] = dist[s] + 1
                    changed = True
        if not changed:
            break
    return -1 if dist[N] == INF else dist[N]

for board, expected in examples:
    got = snakes_bellman_ford(board)
    print(got, "OK" if got == expected else "SAI")
    assert got == expected

4 OK
1 OK


**Độ phức tạp và nút thắt.** Tối đa `n^2` vòng, mỗi vòng `6 n^2` cạnh: `O(n^4)`. Với `n = 20` là `~ 400 * 2400 ~ 10^6`, vẫn nhỏ nhưng nhiều hơn cần thiết: nếu bảng có thang dài thì cần nhiều vòng và mỗi vòng quét lại cả những ô đã ổn định. Hãy xem số vòng và thời gian:

In [6]:
def random_board(rng, n, p_snake=0.15):
    N = n * n
    flat = [0] + [-1] * N
    for s in range(2, N):                                # ô 1 và ô N không có rắn/thang
        if rng.random() < p_snake:
            flat[s] = rng.randint(1, N)
    board = [[-1] * n for _ in range(n)]
    for s in range(1, N + 1):
        r, c = label_to_rc(s, n)
        board[r][c] = flat[s]
    return board

rng = random.Random(909)
for n in (5, 10, 15, 20):
    b = random_board(rng, n)
    t0 = time.perf_counter()
    r = snakes_bellman_ford(b)
    print(f"n={n:2}: kết quả {r:3}, {1000 * (time.perf_counter() - t0):8.2f} ms")

n= 5: kết quả   4,     0.04 ms
n=10: kết quả   4,     0.24 ms
n=15: kết quả   7,     0.58 ms
n=20: kết quả   5,     1.09 ms


Nút thắt: ta cập nhật lặp lại các ô **cũ** nhiều lần. Vì mọi cạnh có độ dài `1`, có thể xử lý ô theo **từng lớp khoảng cách** (BFS) để mỗi ô chỉ được mở rộng **một lần**.

## 6. Ý tưởng thuật toán / bước đột phá

**BFS theo lớp khoảng cách.** Lớp 0 là `{1}`. Lớp `k + 1` gồm mọi ô tới được từ một ô lớp `k` bằng một lượt và chưa nằm ở lớp nào trước đó. Lần đầu một ô xuất hiện chính là khoảng cách ngắn nhất. Mỗi ô được mở rộng đúng một lần nên tổng chi phí `O(6 n^2)`.

Điểm chú ý của đề:

1. Nhãn `-> (hàng, cột)` theo zig-zag (hoặc làm phẳng bảng).
2. Ô `next` có rắn/thang thì **bắt buộc** đi tới đích của nó, và **chỉ một lần** (không đi tiếp chuỗi).
3. Xúc xắc bị cắt bởi `n^2`: `next <= n^2`.

| Cách | Ý tưởng |
|---|---|
| 2 | BFS với hàm đổi nhãn thành `(r, c)` mỗi lần |
| 3 | BFS trên mảng đã **làm phẳng** (nộp) |
| 4 | BFS theo **từng lớp** bằng tập hợp (đếm lớp) |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — BFS với hàm đổi nhãn thành tọa độ

**Giải thích từng đoạn.**

- `dist = {1: 0}` hoặc mảng `dist` và `visited`.
- Lấy `s` từ hàng đợi; với `nxt` từ `s + 1` tới `min(s + 6, n^2)`: tính `(r, c) = label_to_rc(nxt, n)`; `dest = board[r][c]` nếu khác `-1`, ngược lại `dest = nxt`.
- Nếu `dest` chưa thăm: đánh dấu, `dist[dest] = dist[s] + 1`, đẩy vào hàng đợi. Nếu `dest == n^2` có thể trả luôn `dist[dest]`.
- Hàng đợi cạn mà chưa tới `n^2`: trả `-1`.

In [7]:
def snakes_bfs_coords(board: List[List[int]]) -> int:
    n = len(board)
    N = n * n
    dist = [-1] * (N + 1)
    dist[1] = 0
    dq = deque([1])
    while dq:
        s = dq.popleft()
        if s == N:
            return dist[s]
        for nxt in range(s + 1, min(s + 6, N) + 1):
            r, c = label_to_rc(nxt, n)
            dest = board[r][c] if board[r][c] != -1 else nxt
            if dist[dest] == -1:
                dist[dest] = dist[s] + 1
                dq.append(dest)
    return -1

for board, expected in examples:
    got = snakes_bfs_coords(board)
    print(got, "OK" if got == expected else "SAI")
    assert got == expected

4 OK
1 OK


**Code nộp LeetCode (Cách 2):**

In [8]:
class Solution:
    def snakesAndLadders(self, board: List[List[int]]) -> int:
        from collections import deque
        n = len(board)
        N = n * n

        def get(s: int) -> int:
            q, rem = divmod(s - 1, n)
            r = n - 1 - q
            c = rem if q % 2 == 0 else n - 1 - rem
            return board[r][c]

        dist = [-1] * (N + 1)
        dist[1] = 0
        dq = deque([1])
        while dq:
            s = dq.popleft()
            if s == N:
                return dist[s]
            for nxt in range(s + 1, min(s + 6, N) + 1):
                v = get(nxt)
                dest = v if v != -1 else nxt
                if dist[dest] == -1:
                    dist[dest] = dist[s] + 1
                    dq.append(dest)
        return -1

sol_coords = Solution().snakesAndLadders
assert [sol_coords(b) for b, _ in examples] == [e for _, e in examples]
print("Cách 2 qua ví dụ")

Cách 2 qua ví dụ


**Độ phức tạp (Cách 2):** `O(n^2)` thời gian (mỗi ô được mở rộng một lần, mỗi lần 6 cạnh); `O(n^2)` bộ nhớ.

---

### 7.2. Cách 3 — BFS trên mảng đã làm phẳng (cách nộp chính)

**Ý tưởng khác.** Làm phẳng một lần: `flat[s]` là giá trị ô nhãn `s`. Khi đó vòng BFS không còn gọi hàm đổi tọa độ: `dest = flat[nxt] if flat[nxt] != -1 else nxt`.

**Giải thích từng đoạn.**

- `for q, row in enumerate(reversed(board))`: `q = 0` là hàng đáy; hàng `q` lẻ đảo chiều `row[::-1]`.
- `flat = [0] + [...]` để nhãn bắt đầu từ `1`.
- BFS giống Cách 2.

In [9]:
def snakes_bfs_flat(board: List[List[int]]) -> int:
    N = len(board) ** 2
    flat = flatten(board)
    dist = [-1] * (N + 1)
    dist[1] = 0
    dq = deque([1])
    while dq:
        s = dq.popleft()
        if s == N:
            return dist[s]
        for nxt in range(s + 1, min(s + 6, N) + 1):
            dest = flat[nxt] if flat[nxt] != -1 else nxt
            if dist[dest] == -1:
                dist[dest] = dist[s] + 1
                dq.append(dest)
    return -1

for board, expected in examples:
    got = snakes_bfs_flat(board)
    print(got, "OK" if got == expected else "SAI")
    assert got == expected

4 OK
1 OK


**Code nộp LeetCode (Cách 3):**

In [10]:
class Solution:
    def snakesAndLadders(self, board: List[List[int]]) -> int:
        from collections import deque
        n = len(board)
        N = n * n
        flat = [0]
        for q, row in enumerate(reversed(board)):
            flat.extend(row if q % 2 == 0 else row[::-1])
        dist = [-1] * (N + 1)
        dist[1] = 0
        dq = deque([1])
        while dq:
            s = dq.popleft()
            if s == N:
                return dist[s]
            for nxt in range(s + 1, min(s + 6, N) + 1):
                dest = flat[nxt] if flat[nxt] != -1 else nxt
                if dist[dest] == -1:
                    dist[dest] = dist[s] + 1
                    dq.append(dest)
        return -1

sol_flat = Solution().snakesAndLadders
assert [sol_flat(b) for b, _ in examples] == [e for _, e in examples]
print("Cách 3 qua ví dụ")

Cách 3 qua ví dụ


**Độ phức tạp (Cách 3):** `O(n^2)` thời gian và bộ nhớ (mảng `flat`, `dist`, hàng đợi).

---

### 7.3. Cách 4 — BFS theo từng lớp bằng tập hợp

**Ý tưởng khác.** Giữ **tập** các ô ở lớp hiện tại `frontier` và tập `seen`. Mỗi bước tạo `next_frontier` = mọi đích mới từ `frontier`; số lượt tăng `1` khi chuyển sang lớp kế. Gặp `n^2` trong lớp nào thì số lượt là chỉ số lớp đó. Không cần mảng khoảng cách.

**Giải thích từng đoạn.** `steps` là số lượt đã đi để tới các ô của `frontier`; khi `N in frontier` trả `steps`; khi `frontier` rỗng trả `-1`.

In [11]:
def snakes_layers(board: List[List[int]]) -> int:
    N = len(board) ** 2
    flat = flatten(board)
    frontier, seen, steps = {1}, {1}, 0
    while frontier:
        if N in frontier:
            return steps
        nxt_frontier = set()
        for s in frontier:
            for nxt in range(s + 1, min(s + 6, N) + 1):
                dest = flat[nxt] if flat[nxt] != -1 else nxt
                if dest not in seen:
                    seen.add(dest)
                    nxt_frontier.add(dest)
        frontier = nxt_frontier
        steps += 1
    return -1

for board, expected in examples:
    got = snakes_layers(board)
    print(got, "OK" if got == expected else "SAI")
    assert got == expected

4 OK
1 OK


**Code nộp LeetCode (Cách 4):**

In [12]:
class Solution:
    def snakesAndLadders(self, board: List[List[int]]) -> int:
        n = len(board)
        N = n * n
        flat = [0]
        for q, row in enumerate(reversed(board)):
            flat.extend(row if q % 2 == 0 else row[::-1])
        frontier, seen, steps = {1}, {1}, 0
        while frontier:
            if N in frontier:
                return steps
            nxt_frontier = set()
            for s in frontier:
                for nxt in range(s + 1, min(s + 6, N) + 1):
                    dest = flat[nxt] if flat[nxt] != -1 else nxt
                    if dest not in seen:
                        seen.add(dest)
                        nxt_frontier.add(dest)
            frontier = nxt_frontier
            steps += 1
        return -1

sol_layers = Solution().snakesAndLadders
assert [sol_layers(b) for b, _ in examples] == [e for _, e in examples]
print("Cách 4 qua ví dụ")

Cách 4 qua ví dụ


**Độ phức tạp (Cách 4):** `O(n^2)` thời gian và bộ nhớ.

## 8. TRACE từng bước

### 8.1. Cách 3 (BFS làm phẳng) cho ví dụ 1

Ta in từng lớp: các ô được thêm vào hàng đợi, kèm ô đã chọn `nxt` và đích sau rắn/thang.

In [13]:
def trace_bfs(board):
    N = len(board) ** 2
    flat = flatten(board)
    print("ô có rắn/thang:", {s: flat[s] for s in range(1, N + 1) if flat[s] != -1})
    dist = [-1] * (N + 1)
    dist[1] = 0
    dq = deque([1])
    while dq:
        s = dq.popleft()
        if s == N:
            print(f"tới ô {N} sau {dist[s]} lượt")
            return dist[s]
        added = []
        for nxt in range(s + 1, min(s + 6, N) + 1):
            dest = flat[nxt] if flat[nxt] != -1 else nxt
            note = f"{nxt}->{dest}" if dest != nxt else str(nxt)
            if dist[dest] == -1:
                dist[dest] = dist[s] + 1
                dq.append(dest)
                added.append(note)
        if added:
            print(f"lấy ô {s:2} (dist={dist[s]}): thêm {added}")
    return -1

assert trace_bfs(examples[0][0]) == 4

ô có rắn/thang: {2: 15, 14: 35, 17: 13}
lấy ô  1 (dist=0): thêm ['2->15', '3', '4', '5', '6', '7']
lấy ô 15 (dist=1): thêm ['16', '17->13', '18', '19', '20', '21']
lấy ô  3 (dist=1): thêm ['8', '9']
lấy ô  4 (dist=1): thêm ['10']
lấy ô  5 (dist=1): thêm ['11']
lấy ô  6 (dist=1): thêm ['12']
lấy ô 16 (dist=2): thêm ['22']
lấy ô 13 (dist=2): thêm ['14->35']
lấy ô 18 (dist=2): thêm ['23', '24']
lấy ô 19 (dist=2): thêm ['25']
lấy ô 20 (dist=2): thêm ['26']
lấy ô 21 (dist=2): thêm ['27']
lấy ô 22 (dist=3): thêm ['28']
lấy ô 35 (dist=3): thêm ['36']
lấy ô 23 (dist=3): thêm ['29']
lấy ô 24 (dist=3): thêm ['30']
lấy ô 25 (dist=3): thêm ['31']
lấy ô 26 (dist=3): thêm ['32']
lấy ô 27 (dist=3): thêm ['33']
lấy ô 28 (dist=4): thêm ['34']
tới ô 36 sau 4 lượt


### 8.2. Đường đi tối ưu của ví dụ 1 (truy vết bằng `parent`)

Ta lưu `parent[dest] = (s, nxt)` để dựng lại đường: nhãn `s`, ô chọn `nxt`, đích sau rắn/thang.

In [14]:
def shortest_path(board):
    N = len(board) ** 2
    flat = flatten(board)
    dist = [-1] * (N + 1)
    parent = {}
    dist[1] = 0
    dq = deque([1])
    while dq:
        s = dq.popleft()
        if s == N:
            break
        for nxt in range(s + 1, min(s + 6, N) + 1):
            dest = flat[nxt] if flat[nxt] != -1 else nxt
            if dist[dest] == -1:
                dist[dest] = dist[s] + 1
                parent[dest] = (s, nxt)
                dq.append(dest)
    if dist[N] == -1:
        return None
    path, cur = [], N
    while cur != 1:
        s, nxt = parent[cur]
        path.append((s, nxt, cur))
        cur = s
    return path[::-1]

path = shortest_path(examples[0][0])
for s, nxt, dest in path:
    print(f"từ {s} chọn ô {nxt}" + (f", đi theo rắn/thang tới {dest}" if dest != nxt else ""))
assert len(path) == 4

từ 1 chọn ô 2, đi theo rắn/thang tới 15
từ 15 chọn ô 17, đi theo rắn/thang tới 13
từ 13 chọn ô 14, đi theo rắn/thang tới 35
từ 35 chọn ô 36


### 8.3. Quy tắc "chỉ một lần": bảng `[[-1,4],[-1,3]]` của đề

Làm phẳng: `flat = [0, -1, 3, 4, -1]`: ô `2` có thang tới `3`, ô `3` có thang tới `4`. Từ ô `1` có ba lựa chọn: chọn ô `2` đi tới `3` (và **dừng ở `3`**, không đi tiếp lên `4`); chọn ô `3` đi tới `4`; chọn ô `4`. Hai lựa chọn sau tới đích ngay trong **một** lượt, nên đáp án là `1`. Ví dụ này chỉ minh họa quy tắc; ở mục 13 ta tìm bảng mà quy tắc "một lần" **làm thay đổi đáp án**.

In [15]:
tricky = [[-1, 4], [-1, 3]]
flat = flatten(tricky)
print("flat:", flat)
for nxt in (2, 3, 4):
    dest = flat[nxt] if flat[nxt] != -1 else nxt
    print(f"chọn ô {nxt} -> dừng ở ô {dest}")
assert snakes_bfs_flat(tricky) == 1 and Solution().snakesAndLadders(tricky) == 1

flat: [0, -1, 3, 4, -1]
chọn ô 2 -> dừng ở ô 3
chọn ô 3 -> dừng ở ô 4
chọn ô 4 -> dừng ở ô 4


## 9. Code nộp LeetCode

Ta chọn **Cách 3** (BFS trên mảng đã làm phẳng): ngắn gọn, `O(n^2)`, ít chỗ sai nhất vì phần đổi tọa độ chỉ làm **một lần**.

```python
class Solution:
    def snakesAndLadders(self, board: List[List[int]]) -> int:
        from collections import deque
        n = len(board)
        N = n * n
        flat = [0]
        for q, row in enumerate(reversed(board)):
            flat.extend(row if q % 2 == 0 else row[::-1])
        dist = [-1] * (N + 1)
        dist[1] = 0
        dq = deque([1])
        while dq:
            s = dq.popleft()
            if s == N:
                return dist[s]
            for nxt in range(s + 1, min(s + 6, N) + 1):
                dest = flat[nxt] if flat[nxt] != -1 else nxt
                if dist[dest] == -1:
                    dist[dest] = dist[s] + 1
                    dq.append(dest)
        return -1
```

In [16]:
class Solution:
    def snakesAndLadders(self, board: List[List[int]]) -> int:
        from collections import deque
        n = len(board)
        N = n * n
        flat = [0]
        for q, row in enumerate(reversed(board)):
            flat.extend(row if q % 2 == 0 else row[::-1])
        dist = [-1] * (N + 1)
        dist[1] = 0
        dq = deque([1])
        while dq:
            s = dq.popleft()
            if s == N:
                return dist[s]
            for nxt in range(s + 1, min(s + 6, N) + 1):
                dest = flat[nxt] if flat[nxt] != -1 else nxt
                if dist[dest] == -1:
                    dist[dest] = dist[s] + 1
                    dq.append(dest)
        return -1

sol = Solution()
for board, expected in examples:
    got = sol.snakesAndLadders(board)
    print(f"n={len(board)}: {got}", "OK" if got == expected else "SAI")
    assert got == expected

n=6: 4 OK
n=2: 1 OK


## 10. Vì sao đúng

**(a) Mô hình đồ thị.** Đỉnh là các nhãn `1..N` (`N = n^2`). Từ đỉnh `s < N`, với mỗi `nxt` trong `[s + 1, min(s + 6, N)]` có đúng một cạnh tới `f(nxt)`, trong đó `f(x) = flat[x]` nếu ô `x` có rắn/thang và `f(x) = x` ngược lại. Đúng quy tắc của đề: ô `nxt` có rắn/thang thì **bắt buộc** đi tới đích của nó, và chỉ **một lần** (ta **không** áp dụng `f` lần thứ hai lên đích).

**(b) BFS tối ưu.** Mọi cạnh có độ dài `1`, nên BFS gán cho mỗi đỉnh khoảng cách ngắn nhất từ `1`: bằng quy nạp theo lớp, đỉnh đạt được lần đầu ở bước `k` không thể đạt ở bước nhỏ hơn (nếu đạt được ở bước `j < k` nó đã được gán từ lớp `j`). Ta đánh dấu theo **đích sau rắn/thang** vì đó là trạng thái thực sự của người chơi.

**(c) Đổi nhãn.** Hàng đáy chạy trái sang phải, hàng kế chạy phải sang trái, ... Việc đảo `row[::-1]` ở các hàng `q` lẻ (tính từ đáy) cho đúng thứ tự nhãn.

**(d) Trả về `-1`.** Nếu hàng đợi cạn mà chưa gặp `N` thì `N` không thuộc thành phần liên thông có hướng của `1`, trả `-1`.

**(e) Cách 1.** Bellman-Ford: sau `k` vòng, `dist[v]` không lớn hơn độ dài đường đi ngắn nhất dùng tối đa `k` cạnh; dừng khi không đổi (hoặc sau `N` vòng, vì đường ngắn nhất không lặp đỉnh nên có tối đa `N - 1` cạnh).

**Kiểm chứng bằng máy.** Hai oracle **độc lập với BFS**: (1) **Bellman-Ford** ở trên (dùng bảng làm phẳng); (2) **duyệt vét cạn mọi dãy lượt đi** theo định nghĩa gốc, lấy chỉ số `(hàng, cột)` bằng hàm đổi tọa độ **riêng** (không dùng làm phẳng) trên bảng nhỏ (`n <= 3`, độ sâu giới hạn) để kiểm tra luôn việc đổi nhãn. Hai oracle phải khớp nhau và khớp mọi lời giải.

In [17]:
def oracle_dfs(board: List[List[int]]) -> int:
    """Vét cạn mọi dãy lượt đi (độ sâu giới hạn N - 1), dùng toạ độ trực tiếp (không làm phẳng)."""
    n = len(board)
    N = n * n
    # dựng bản đồ nhãn -> (r, c) bằng cách đi zig-zag tường minh từ góc dưới trái
    pos = {}
    label = 1
    for q in range(n):
        r = n - 1 - q
        cols = range(n) if q % 2 == 0 else range(n - 1, -1, -1)
        for c in cols:
            pos[label] = (r, c)
            label += 1
    best = [INF]

    def go(s, steps):
        if steps >= best[0] or steps > N - 1:
            return
        for nxt in range(s + 1, min(s + 6, N) + 1):
            r, c = pos[nxt]
            dest = board[r][c] if board[r][c] != -1 else nxt
            if dest == N:
                best[0] = min(best[0], steps + 1)
            else:
                go(dest, steps + 1)

    go(1, 0)
    return -1 if best[0] == INF else best[0]

rng = random.Random(1909)
for _ in range(300):
    n = rng.randint(2, 3)
    b = random_board(rng, n, p_snake=rng.choice([0.0, 0.3, 0.6]))
    assert oracle_dfs(b) == snakes_bellman_ford(b), b
print("oracle vét cạn (toạ độ zig-zag tường minh) và Bellman-Ford khớp nhau trên 300 bảng nhỏ")

oracle vét cạn (toạ độ zig-zag tường minh) và Bellman-Ford khớp nhau trên 300 bảng nhỏ


## 11. Time / Space Complexity

Gọi `N = n^2` là số ô.

| Cách | Time | Space |
|---|---|---|
| 1. Bellman-Ford | `O(N^2 * 6)` | `O(N)` |
| 2. BFS + đổi tọa độ | `O(N)` | `O(N)` |
| 3. BFS làm phẳng | `O(N)` | `O(N)` |
| 4. BFS theo lớp (tập) | `O(N)` | `O(N)` |

Đo thử với `n = 20` (cỡ tối đa) và nhiều bảng:

In [18]:
rng = random.Random(11)
boards = [random_board(rng, 20, p) for p in (0.0, 0.05, 0.15, 0.3) for _ in range(30)]
ref = [snakes_bellman_ford(b) for b in boards]
for name, fn in (("Cách 2 đổi tọa độ", sol_coords), ("Cách 3 làm phẳng", sol_flat), ("Cách 4 lớp", sol_layers), ("nộp", Solution().snakesAndLadders)):
    t0 = time.perf_counter()
    res = [fn(b) for b in boards]
    dt = (time.perf_counter() - t0) * 1000
    assert res == ref, name
    print(f"{name:18} {len(boards)} bảng 20x20: {dt:8.1f} ms")
t0 = time.perf_counter()
[snakes_bellman_ford(b) for b in boards]
print(f"{'Bellman-Ford':18} {len(boards)} bảng 20x20: {1000 * (time.perf_counter() - t0):8.1f} ms")
print("số bảng không tới được ô cuối:", sum(1 for r in ref if r == -1), "/", len(ref))

Cách 2 đổi tọa độ  120 bảng 20x20:     61.8 ms


Cách 3 làm phẳng   120 bảng 20x20:     24.1 ms
Cách 4 lớp         120 bảng 20x20:     25.1 ms
nộp                120 bảng 20x20:     23.6 ms
Bellman-Ford       120 bảng 20x20:    103.4 ms
số bảng không tới được ô cuối: 0 / 120


## 12. Bộ test

Gồm: ví dụ của đề, edge case, phản ví dụ cho các cách sai (tìm bằng tìm kiếm có seed), đối chiếu mọi cách với hai oracle (Bellman-Ford và vét cạn toạ độ trực tiếp), test ngẫu nhiên có seed (đúng ràng buộc: ô `1` và ô `n^2` không có rắn/thang) và tính chất.

In [19]:
def solutions_list():
    return {
        "bfs_coords": snakes_bfs_coords,
        "bfs_flat": snakes_bfs_flat,
        "layers": snakes_layers,
        "sol_coords": sol_coords,
        "sol_flat": sol_flat,
        "sol_layers": sol_layers,
        "final": Solution().snakesAndLadders,
    }

def run_all(board, expected=None):
    ref = snakes_bellman_ford(board)
    if len(board) <= 3:
        assert oracle_dfs(board) == ref, ("oracle lệch", board)
    for name, fn in solutions_list().items():
        assert fn([row[:] for row in board]) == ref, (name, board)
    if expected is not None:
        assert ref == expected, ("kỳ vọng sai", board, expected, ref)
    return ref

def board_from_flat(n, flat_map):
    """Dựng bảng n x n từ dict nhãn -> đích (còn lại là -1); dùng hàm đổi toạ độ riêng."""
    board = [[-1] * n for _ in range(n)]
    for s, d in flat_map.items():
        r, c = label_to_rc(s, n)
        board[r][c] = d
    return board

# 12.1 ví dụ đề
for board, expected in examples:
    assert run_all(board, expected) == expected
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay, kèm lý do)
edge = [
    ([[-1, -1], [-1, -1]], 1),                              # n = 2, không rắn/thang: 1 -> 4 (chọn ô 4 ngay)
    ([[-1, -1], [-1, 3]], 1),                               # ví dụ 2
    ([[-1, 4], [-1, 3]], 1),                                # chọn ô 3 (thang lên 4) hoặc ô 4: 1 lượt
    ([[-1, -1, -1], [-1, -1, -1], [-1, -1, -1]], 2),        # n = 3, N = 9: 1 -> 7 -> 9
    (board_from_flat(3, {2: 8}), 2),                        # thang 2 -> 8, rồi 8 -> 9
    (board_from_flat(3, {2: 1, 3: 1, 4: 1, 5: 1, 6: 1, 7: 1}), -1),   # các ô 2..7 đều là rắn về 1: không bao giờ tới ô 8, 9
    (board_from_flat(4, {2: 14}), 2),                       # thang 2 -> 14, rồi 14 -> 16: 2 lượt (không thang: 3 lượt)
    (board_from_flat(4, {}), 3),                            # n = 4, N = 16: 1 -> 7 -> 13 -> 16
    (board_from_flat(5, {}), 4),                            # N = 25: 1 -> 7 -> 13 -> 19 -> 25
    (board_from_flat(6, {}), 6),                            # N = 36: 1,7,13,19,25,31,36 = 6 lượt
]
for board, expected in edge:
    assert run_all(board, expected) == expected, board
print("12.2 edge case: OK", len(edge), "ca")

12.1 ví dụ đề: OK


12.2 edge case: OK 10 ca


In [20]:
# 12.3 phản ví dụ: các cách sai (tìm bằng tìm kiếm có seed rồi xác nhận lời giải đúng không bị lỗi)
def wrong_chain(board):                             # đi theo chuỗi thang liên tiếp trong cùng một lượt
    N = len(board) ** 2
    flat = flatten(board)
    def f(x):
        seen = set()
        while flat[x] != -1 and x not in seen:
            seen.add(x)
            x = flat[x]
        return x
    dist = [-1] * (N + 1)
    dist[1] = 0
    dq = deque([1])
    while dq:
        s = dq.popleft()
        if s == N:
            return dist[s]
        for nxt in range(s + 1, min(s + 6, N) + 1):
            dest = f(nxt)
            if dist[dest] == -1:
                dist[dest] = dist[s] + 1
                dq.append(dest)
    return -1

def wrong_no_zigzag(board):                         # quên đảo chiều các hàng lẻ khi làm phẳng
    N = len(board) ** 2
    flat = [0] + [v for row in reversed(board) for v in row]
    dist = [-1] * (N + 1)
    dist[1] = 0
    dq = deque([1])
    while dq:
        s = dq.popleft()
        if s == N:
            return dist[s]
        for nxt in range(s + 1, min(s + 6, N) + 1):
            dest = flat[nxt] if flat[nxt] != -1 else nxt
            if dist[dest] == -1:
                dist[dest] = dist[s] + 1
                dq.append(dest)
    return -1

def wrong_greedy_max(board):                        # tham lam: luôn đi xa nhất có thể
    N = len(board) ** 2
    flat = flatten(board)
    s, steps = 1, 0
    visited = set()
    while s != N:
        if s in visited:
            return -1
        visited.add(s)
        nxt = min(s + 6, N)
        s = flat[nxt] if flat[nxt] != -1 else nxt
        steps += 1
    return steps

def wrong_dfs_first(board):                         # DFS có visited: trả về đường đầu tiên tìm thấy
    N = len(board) ** 2
    flat = flatten(board)
    visited = {1}
    def dfs(s, steps):
        if s == N:
            return steps
        for nxt in range(s + 1, min(s + 6, N) + 1):
            dest = flat[nxt] if flat[nxt] != -1 else nxt
            if dest not in visited:
                visited.add(dest)
                r = dfs(dest, steps + 1)
                if r != -1:
                    return r
        return -1
    return dfs(1, 0)

def wrong_no_cap(board):                            # quên cắt nxt <= N: lỗi khi s gần cuối
    N = len(board) ** 2
    flat = flatten(board)
    dist = [-1] * (N + 1)
    dist[1] = 0
    dq = deque([1])
    while dq:
        s = dq.popleft()
        if s == N:
            return dist[s]
        for nxt in range(s + 1, s + 7):
            if nxt > N:
                nxt = N + 1                           # sẽ chỉ số ngoài mảng flat
            dest = flat[nxt] if flat[nxt] != -1 else nxt
            if dist[dest] == -1:
                dist[dest] = dist[s] + 1
                dq.append(dest)
    return -1

def attempt(fn, board):
    try:
        return fn([row[:] for row in board])
    except Exception as e:
        return f"lỗi {type(e).__name__}"

def find_counter(fn, tries=20000, seed=3):
    rng = random.Random(seed)
    for _ in range(tries):
        n = rng.randint(2, 5)
        b = random_board(rng, n, p_snake=rng.choice([0.1, 0.3, 0.5]))
        if attempt(fn, b) != snakes_bellman_ford(b):
            return b
    return None

counters = {}
for name, fn in (("đi theo chuỗi thang", wrong_chain), ("quên đảo chiều hàng lẻ", wrong_no_zigzag),
                 ("tham lam đi xa nhất", wrong_greedy_max), ("DFS lấy đường đầu tiên", wrong_dfs_first),
                 ("quên cắt ở n^2", wrong_no_cap)):
    b = find_counter(fn)
    assert b is not None, name
    counters[name] = b
    print(f"{name:26} n={len(b)}: sai -> {attempt(fn, b)}, đúng -> {snakes_bellman_ford(b)}")
    run_all(b)
print("12.3 phản ví dụ: OK")

đi theo chuỗi thang        n=5: sai -> 3, đúng -> 4
quên đảo chiều hàng lẻ     n=4: sai -> -1, đúng -> 3
tham lam đi xa nhất        n=4: sai -> 2, đúng -> 1
DFS lấy đường đầu tiên     n=3: sai -> 3, đúng -> 1
quên cắt ở n^2             n=3: sai -> lỗi IndexError, đúng -> 1
12.3 phản ví dụ: OK


In [21]:
# 12.4 ngẫu nhiên nhỏ: n = 2..3, so với oracle vét cạn toạ độ và Bellman-Ford
rng = random.Random(4909)
for _ in range(1200):
    n = rng.randint(2, 3)
    b = random_board(rng, n, p_snake=rng.choice([0.0, 0.2, 0.4, 0.7]))
    run_all(b)
print("12.4 ngẫu nhiên n <= 3 (1200 ca, gồm oracle vét cạn): OK")

# 12.5 ngẫu nhiên n = 2..20 (đúng ràng buộc), so với Bellman-Ford
rng = random.Random(5909)
unreachable = 0
for _ in range(500):
    n = rng.randint(2, 20)
    b = random_board(rng, n, p_snake=rng.choice([0.0, 0.05, 0.15, 0.3, 0.5]))
    unreachable += run_all(b) == -1
print(f"12.5 ngẫu nhiên n tới 20 (500 ca): OK, {unreachable} bảng không tới được ô cuối")

# 12.6 các bảng đặc biệt: toàn thang lên cuối chuỗi, toàn rắn về 1, hàng rào không vượt được
special = {}
n = 6
N = n * n
special["mọi ô 2..7 về 1 (chặn hoàn toàn)"] = board_from_flat(n, {s: 1 for s in range(2, 8)})
special["một thang 2 -> N - 1"] = board_from_flat(n, {2: N - 1})
special["thang 2 -> 3 -> 4 -> ... (chuỗi)"] = board_from_flat(n, {s: s + 1 for s in range(2, N - 1)})
special["rắn xuống ô 1 ở cuối"] = board_from_flat(n, {N - 1: 1, N - 2: 1, N - 3: 1})
for name, b in special.items():
    print(f"  {name:36}: {run_all(b)}")
print("12.6 bảng đặc biệt: OK")

# 12.7 tính chất: thêm rắn/thang không thể làm đáp án nhỏ hơn số lượt tối thiểu theo khoảng cách: >= ceil((N - 1) / 6) khi không có thang
rng = random.Random(7909)
import math
for _ in range(300):
    n = rng.randint(2, 10)
    plain = board_from_flat(n, {})
    N = n * n
    assert Solution().snakesAndLadders(plain) == math.ceil((N - 1) / 6)
    b = random_board(rng, n, 0.2)
    got = Solution().snakesAndLadders(b)
    assert got == -1 or got >= 1
print("12.7 tính chất (bảng trơn cần ceil((N - 1) / 6) lượt): OK")

12.4 ngẫu nhiên n <= 3 (1200 ca, gồm oracle vét cạn): OK


12.5 ngẫu nhiên n tới 20 (500 ca): OK, 0 bảng không tới được ô cuối
  mọi ô 2..7 về 1 (chặn hoàn toàn)    : -1
  một thang 2 -> N - 1                : 2
  thang 2 -> 3 -> 4 -> ... (chuỗi)    : 6
  rắn xuống ô 1 ở cuối                : 6
12.6 bảng đặc biệt: OK
12.7 tính chất (bảng trơn cần ceil((N - 1) / 6) lượt): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Đi theo chuỗi thang liên tiếp trong cùng một lượt

Đề nói rõ: mỗi lượt chỉ đi theo rắn/thang **một lần**; nếu đích lại là điểm bắt đầu của rắn/thang khác thì **không** đi tiếp.

In [22]:
b = counters["đi theo chuỗi thang"]
print("flat:", flatten(b))
print("sai ->", attempt(wrong_chain, b), "| đúng ->", Solution().snakesAndLadders(b))
assert attempt(wrong_chain, b) != Solution().snakesAndLadders(b)

flat: [0, -1, -1, 10, -1, 12, -1, -1, -1, -1, -1, -1, 17, -1, -1, -1, -1, -1, 7, -1, -1, -1, -1, -1, -1, -1]
sai -> 3 | đúng -> 4


Vì sao sai: áp dụng `f` nhiều lần cho phép nhảy qua nhiều rắn/thang trong một lượt, rút ngắn đáp án một cách trái luật. **Đúng:** `dest = flat[nxt] if flat[nxt] != -1 else nxt` (một lần).

### Lỗi 2 — Quên đảo chiều các hàng lẻ (zig-zag)

Làm phẳng đơn giản `row` theo thứ tự cho nhãn sai ở hàng đảo chiều.

In [23]:
b = counters["quên đảo chiều hàng lẻ"]
print("sai ->", attempt(wrong_no_zigzag, b), "| đúng ->", Solution().snakesAndLadders(b))
assert attempt(wrong_no_zigzag, b) != Solution().snakesAndLadders(b)
print("flat đúng ví dụ 1 ở ô 2, 17, 14:", flatten(examples[0][0])[2], flatten(examples[0][0])[17], flatten(examples[0][0])[14])

sai -> -1 | đúng -> 3
flat đúng ví dụ 1 ở ô 2, 17, 14: 15 13 35


Vì sao sai: nhãn `n + 1` nằm ở **cột phải** của hàng thứ hai từ đáy, không phải cột trái. **Đúng:** `row if q % 2 == 0 else row[::-1]` với `q` là chỉ số hàng tính từ đáy.

### Lỗi 3 — Tham lam "luôn đi xa nhất"

Đi xa nhất có thể rơi vào rắn hoặc bỏ lỡ thang ngắn hơn.

In [24]:
b = counters["tham lam đi xa nhất"]
print("sai ->", attempt(wrong_greedy_max, b), "| đúng ->", Solution().snakesAndLadders(b))
assert attempt(wrong_greedy_max, b) != Solution().snakesAndLadders(b)

sai -> 2 | đúng -> 1


Vì sao sai: mục tiêu là số lượt **tối thiểu** toàn cục, không phải lợi ích cục bộ từng lượt. **Đúng:** BFS khám phá mọi lựa chọn theo lớp.

### Lỗi 4 — DFS thay cho BFS

DFS trả về đường **đầu tiên tìm thấy**, không đảm bảo ngắn nhất.

In [25]:
b = counters["DFS lấy đường đầu tiên"]
print("sai ->", attempt(wrong_dfs_first, b), "| đúng ->", Solution().snakesAndLadders(b))
assert attempt(wrong_dfs_first, b) != Solution().snakesAndLadders(b)

sai -> 3 | đúng -> 1


Vì sao sai: DFS đi sâu theo xúc xắc nhỏ trước và có thể tới đích bằng đường dài. **Đúng:** BFS (lớp khoảng cách tăng dần) cho đường ngắn nhất khi cạnh có độ dài `1`.

### Lỗi 5 — Quên cắt ở `n^2`

`range(s + 1, s + 7)` mà không cắt ở `N` vượt quá mảng khi `s` gần cuối.

In [26]:
b = counters["quên cắt ở n^2"]
print("sai ->", attempt(wrong_no_cap, b), "| đúng ->", Solution().snakesAndLadders(b))
assert attempt(wrong_no_cap, b) != Solution().snakesAndLadders(b)

sai -> lỗi IndexError | đúng -> 1


Vì sao sai: nhãn tối đa là `N`; truy cập `flat[nxt]` với `nxt > N` là ngoài mảng (hoặc dẫn tới ô không tồn tại). **Đúng:** `min(s + 6, N)`.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "số bước/lượt ít nhất" trên đồ thị ẩn với cạnh độ dài đều nhau ⇒ **BFS**.
- **Đổi nhãn zig-zag:** `q, rem = divmod(s - 1, n)`; hàng `n - 1 - q`; cột `rem` nếu `q` chẵn, ngược lại `n - 1 - rem`; hoặc làm phẳng bảng một lần.
- **Trạng thái BFS** là ô **sau** khi áp dụng rắn/thang (đúng một lần).
- Đánh dấu `visited/dist` ngay khi đẩy vào hàng đợi để mỗi ô vào tối đa một lần.
- Cùng họ: Word Ladder (127), Minimum Genetic Mutation (433), Open the Lock (752), Shortest Path in Binary Matrix (1091), Jump Game III (1306).
- Test nên có: `n = 2` (ví dụ nhỏ), thang chuỗi (quy tắc "một lần"), bảng không có rắn/thang (`ceil((N - 1) / 6)`), bảng bị chặn (đáp án `-1`).

## 15. Biến thể

1. **Truy vết đường đi:** lưu `parent` (đã làm ở mục 8).
2. **Đếm số cách đi ngắn nhất** (bài tập 5): đếm số dãy ô chọn đạt số lượt tối thiểu.
3. **Xúc xắc `k` mặt** thay cho 6: đổi hằng số 6.
4. **Bất kỳ chuỗi thang** (đi theo liên tiếp): đổi hàm `f` lặp đến khi không còn rắn/thang (chú ý vòng lặp).
5. Bài liên quan: Jump Game II (45), Minimum Jumps to Reach Home (1654).

Dưới đây là biến thể 3 (xúc xắc `k` mặt) kiểm bằng Bellman-Ford tổng quát.

In [27]:
def snakes_k_sided(board, k):
    N = len(board) ** 2
    flat = flatten(board)
    dist = [-1] * (N + 1)
    dist[1] = 0
    dq = deque([1])
    while dq:
        s = dq.popleft()
        if s == N:
            return dist[s]
        for nxt in range(s + 1, min(s + k, N) + 1):
            dest = flat[nxt] if flat[nxt] != -1 else nxt
            if dist[dest] == -1:
                dist[dest] = dist[s] + 1
                dq.append(dest)
    return -1

def bellman_k(board, k):
    N = len(board) ** 2
    flat = flatten(board)
    dist = [INF] * (N + 1)
    dist[1] = 0
    for _ in range(N):
        for s in range(1, N):
            if dist[s] < INF:
                for nxt in range(s + 1, min(s + k, N) + 1):
                    dest = flat[nxt] if flat[nxt] != -1 else nxt
                    dist[dest] = min(dist[dest], dist[s] + 1)
    return -1 if dist[N] == INF else dist[N]

rng = random.Random(15)
for _ in range(300):
    n = rng.randint(2, 7)
    b = random_board(rng, n, 0.2)
    k = rng.randint(1, 8)
    assert snakes_k_sided(b, k) == bellman_k(b, k), (b, k)
    if k == 6:
        assert snakes_k_sided(b, 6) == Solution().snakesAndLadders(b)
print("xúc xắc 6 mặt = bài gốc:", snakes_k_sided(examples[0][0], 6), "| 2 mặt:", snakes_k_sided(examples[0][0], 2))

xúc xắc 6 mặt = bài gốc: 4 | 2 mặt: 4


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Bellman-Ford | Lan truyền nhiều vòng | `O(N^2)` | `O(N)` | Oracle, đồ thị có trọng số bất kỳ |
| 2. BFS + đổi tọa độ | BFS cổ điển | `O(N)` | `O(N)` | Khi không muốn làm phẳng |
| 3. BFS làm phẳng | BFS trên `flat` | `O(N)` | `O(N)` | **Cách nộp chính** |
| 4. BFS theo lớp | Tập frontier | `O(N)` | `O(N)` | Không cần mảng `dist` |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output ba dòng, rồi chạy để kiểm tra:

```python
print(label_to_rc(1, 3), label_to_rc(4, 3), label_to_rc(9, 3))
print(flatten([[-1, 3], [-1, -1]]))
print(divmod(9 - 1, 3))
```

In [28]:
# Bài 1: ghi dự đoán của bạn
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Bảng `n = 4` (16 ô) chỉ có **một** thang: từ ô `2` lên ô `14` (tức `board[3][1] = 14`). Số lượt tối thiểu là bao nhiêu? Và nếu không có thang thì bao nhiêu?

In [29]:
# Bài 2: điền kết quả tính tay
hand_with_ladder = None
hand_without_ladder = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành BFS trên mảng đã làm phẳng.

In [30]:
def exercise_3_bfs(board: List[List[int]]) -> int:
    N = len(board) ** 2
    flat = flatten(board)
    dist = [-1] * (N + 1)
    dist[1] = 0
    dq = deque([1])
    while dq:
        s = dq.popleft()
        if s == N:
            return dist[s]
        # TODO 1: duyệt nxt từ s + 1 tới min(s + 6, N)
        # TODO 2: dest là đích sau rắn/thang (nếu có), ngược lại là nxt
        # TODO 3: nếu dest chưa thăm thì cập nhật dist[dest] và đưa vào hàng đợi
    return -1


# for board, expected in examples:
#     assert exercise_3_bfs(board) == expected

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 2, hãy tự viết `exercise_4_coords(board)`: BFS dùng hàm đổi nhãn thành `(hàng, cột)` thay vì làm phẳng.

In [31]:
def exercise_4_coords(board: List[List[int]]) -> int:
    # TODO
    pass


# for board, expected in examples:
#     assert exercise_4_coords(board) == expected

### Bài 5 — Mini challenge: đếm số cách đi ngắn nhất (khó)

Viết `exercise_5_ways(board)` trả về **số dãy các ô được chọn** (mỗi lượt chọn `nxt`, không phải đích sau rắn/thang) dẫn tới `n^2` đúng bằng số lượt **tối thiểu**; trả `0` nếu không tới được.

```python
exercise_5_ways([[-1, -1], [-1, 3]])   -> 1     # chỉ có dãy chọn ô 4 ngay lượt đầu
exercise_5_ways([[-1, 4], [-1, 3]])    -> 2     # 1 lượt: chọn ô 3 (lên 4) hoặc chọn ô 4
exercise_5_ways(bảng 3x3 không rắn/thang) -> 5  # 2 lượt: 1 -> k -> 9 với k = 3..7
```
Gợi ý: BFS đếm số đường: `ways[dest] += ways[s]` khi `dist[dest] == dist[s] + 1`.

In [32]:
def exercise_5_ways(board: List[List[int]]) -> int:
    # TODO
    pass


# assert exercise_5_ways([[-1, -1], [-1, 3]]) == 1
# assert exercise_5_ways([[-1, 4], [-1, 3]]) == 2
# assert exercise_5_ways(board_from_flat(3, {})) == 5

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `label_to_rc(s, n)`: `q, rem = divmod(s - 1, n)`.
- Mức 2: `n = 3`: ô 1 ở hàng đáy trái `(2, 0)`; ô 4 ở hàng giữa, cột phải `(1, 2)`; ô 9 ở hàng trên cùng, cột phải `(0, 2)`.
- Mức 3: `(2, 0) (1, 2) (0, 2)`, `[0, -1, -1, 3, -1]`, `(2, 2)`.

**Bài 2**
- Mức 1: không có thang: `1 -> 7 -> 13 -> 16` (mỗi lượt tối đa 6).
- Mức 2: có thang: chọn ô `2` (lên `14`), rồi `14 -> 16` (cần `2`).
- Mức 3: `2` lượt có thang, `3` lượt không có.

**Bài 3**
- Mức 1: `for nxt in range(s + 1, min(s + 6, N) + 1):`.
- Mức 2: `dest = flat[nxt] if flat[nxt] != -1 else nxt`.
- Mức 3: `if dist[dest] == -1: dist[dest] = dist[s] + 1; dq.append(dest)`.

**Bài 4**
- Mức 1: viết hàm đổi `s` thành `(r, c)` bằng `divmod`.
- Mức 2: `dest = board[r][c] if board[r][c] != -1 else nxt`.
- Mức 3: phần còn lại giống Bài 3.

**Bài 5**
- Mức 1: BFS có `dist` và `ways`.
- Mức 2: khi gặp `dest` lần đầu: `dist[dest] = dist[s] + 1`, `ways[dest] = ways[s]`, đưa vào hàng đợi; khi gặp lại với `dist[dest] == dist[s] + 1`: `ways[dest] += ways[s]`.
- Mức 3: kết quả `ways[N]` nếu `dist[N] != -1`, ngược lại `0`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [33]:
print(label_to_rc(1, 3), label_to_rc(4, 3), label_to_rc(9, 3))
print(flatten([[-1, 3], [-1, -1]]))
print(divmod(9 - 1, 3))

assert (label_to_rc(1, 3), label_to_rc(4, 3), label_to_rc(9, 3)) == ((2, 0), (1, 2), (0, 2))
assert flatten([[-1, 3], [-1, -1]]) == [0, -1, -1, 3, -1]
assert divmod(9 - 1, 3) == (2, 2)

(2, 0) (1, 2) (0, 2)
[0, -1, -1, 3, -1]
(2, 2)


### Đáp án Bài 2

Có thang: `2` lượt (`1 -> 2` lên `14`, rồi `14 -> 16`). Không thang: `3` lượt.

In [34]:
hand_with_ladder_solution = 2
hand_without_ladder_solution = 3
b_with = board_from_flat(4, {2: 14})
b_without = board_from_flat(4, {})
assert b_with[3][1] == 14                       # đúng vị trí board[3][1]
assert Solution().snakesAndLadders(b_with) == hand_with_ladder_solution == snakes_bellman_ford(b_with)
assert Solution().snakesAndLadders(b_without) == hand_without_ladder_solution == snakes_bellman_ford(b_without)
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [35]:
def exercise_3_bfs_solution(board: List[List[int]]) -> int:
    N = len(board) ** 2
    flat = flatten(board)
    dist = [-1] * (N + 1)
    dist[1] = 0
    dq = deque([1])
    while dq:
        s = dq.popleft()
        if s == N:
            return dist[s]
        for nxt in range(s + 1, min(s + 6, N) + 1):
            dest = flat[nxt] if flat[nxt] != -1 else nxt
            if dist[dest] == -1:
                dist[dest] = dist[s] + 1
                dq.append(dest)
    return -1

for board, expected in examples:
    assert exercise_3_bfs_solution(board) == expected
rng = random.Random(3)
for _ in range(300):
    b = random_board(rng, rng.randint(2, 8), rng.choice([0.0, 0.2, 0.4]))
    assert exercise_3_bfs_solution(b) == snakes_bellman_ford(b)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [36]:
def exercise_4_coords_solution(board: List[List[int]]) -> int:
    n = len(board)
    N = n * n

    def rc(s):
        q, rem = divmod(s - 1, n)
        return n - 1 - q, (rem if q % 2 == 0 else n - 1 - rem)

    dist = [-1] * (N + 1)
    dist[1] = 0
    dq = deque([1])
    while dq:
        s = dq.popleft()
        if s == N:
            return dist[s]
        for nxt in range(s + 1, min(s + 6, N) + 1):
            r, c = rc(nxt)
            dest = board[r][c] if board[r][c] != -1 else nxt
            if dist[dest] == -1:
                dist[dest] = dist[s] + 1
                dq.append(dest)
    return -1

for board, expected in examples:
    assert exercise_4_coords_solution(board) == expected
rng = random.Random(4)
for _ in range(300):
    b = random_board(rng, rng.randint(2, 8), rng.choice([0.0, 0.2, 0.4]))
    assert exercise_4_coords_solution(b) == snakes_bellman_ford(b)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

Đếm đường đi ngắn nhất bằng BFS: `ways[dest] += ways[s]` khi `dist[dest] == dist[s] + 1`. Với `[[-1,4],[-1,3]]` có `2` dãy (chọn ô `3` hoặc ô `4`).

In [37]:
def exercise_5_ways_solution(board: List[List[int]]) -> int:
    N = len(board) ** 2
    flat = flatten(board)
    dist = [-1] * (N + 1)
    ways = [0] * (N + 1)
    dist[1], ways[1] = 0, 1
    dq = deque([1])
    while dq:
        s = dq.popleft()
        for nxt in range(s + 1, min(s + 6, N) + 1):
            dest = flat[nxt] if flat[nxt] != -1 else nxt
            if dist[dest] == -1:
                dist[dest] = dist[s] + 1
                ways[dest] = ways[s]
                dq.append(dest)
            elif dist[dest] == dist[s] + 1:
                ways[dest] += ways[s]
    return ways[N] if dist[N] != -1 else 0

def ways_oracle(board: List[List[int]]) -> int:
    """Vét cạn mọi dãy ô chọn có độ dài đúng bằng đáp án tối thiểu (dùng toạ độ trực tiếp)."""
    best = snakes_bellman_ford(board)
    if best == -1:
        return 0
    n = len(board)
    N = n * n
    pos, label = {}, 1
    for q in range(n):
        cols = range(n) if q % 2 == 0 else range(n - 1, -1, -1)
        for c in cols:
            pos[label] = (n - 1 - q, c)
            label += 1
    def count(s, left):
        if left == 0:
            return 1 if s == N else 0
        total = 0
        for nxt in range(s + 1, min(s + 6, N) + 1):
            r, c = pos[nxt]
            dest = board[r][c] if board[r][c] != -1 else nxt
            total += count(dest, left - 1)
        return total
    return count(1, best)

assert exercise_5_ways_solution([[-1, -1], [-1, 3]]) == 1 == ways_oracle([[-1, -1], [-1, 3]])
assert exercise_5_ways_solution([[-1, 4], [-1, 3]]) == 2 == ways_oracle([[-1, 4], [-1, 3]])
assert exercise_5_ways_solution(board_from_flat(3, {})) == 5 == ways_oracle(board_from_flat(3, {}))
rng = random.Random(5)
for _ in range(300):
    b = random_board(rng, rng.randint(2, 4), rng.choice([0.0, 0.2, 0.4]))
    assert exercise_5_ways_solution(b) == ways_oracle(b), b
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (BFS trên mảng đã làm phẳng)

```python
class Solution:
    def snakesAndLadders(self, board: List[List[int]]) -> int:
        from collections import deque
        n = len(board)
        N = n * n
        flat = [0]                                           # nhãn bắt đầu từ 1
        for q, row in enumerate(reversed(board)):            # q = 0 là hàng đáy
            flat.extend(row if q % 2 == 0 else row[::-1])    # hàng lẻ đi phải sang trái
        dist = [-1] * (N + 1)
        dist[1] = 0
        dq = deque([1])
        while dq:
            s = dq.popleft()
            if s == N:
                return dist[s]
            for nxt in range(s + 1, min(s + 6, N) + 1):      # xúc xắc, cắt ở N
                dest = flat[nxt] if flat[nxt] != -1 else nxt # rắn/thang: đúng MỘT lần
                if dist[dest] == -1:
                    dist[dest] = dist[s] + 1
                    dq.append(dest)
        return -1
```

Công thức nhớ nhanh: BFS từ `1`; cạnh `s -> f(nxt)` với `nxt in [s + 1, min(s + 6, N)]`; `f` áp dụng một lần.

### Checklist trước khi submit

- [ ] Làm phẳng đúng zig-zag (đảo chiều các hàng lẻ tính từ đáy)?
- [ ] Rắn/thang áp dụng **một lần** mỗi lượt (không đi theo chuỗi)?
- [ ] Cắt `min(s + 6, N)`; `visited/dist` đánh dấu theo **đích sau rắn/thang**?
- [ ] Dùng BFS (không DFS/tham lam) và trả `-1` khi không tới được?
- [ ] Đã thử: `n = 2`, thang chuỗi, bảng không rắn/thang, bảng bị chặn, `n = 20`?
- [ ] Tên method đúng `snakesAndLadders(self, board)` và trả `int`?